# Ứng dụng trích xuất phản hồi sinh viên có chấm điểm

Phòng đào tạo nhận hàng nghìn phản hồi mỗi học kỳ và cần một bảng: phản hồi nói về gì, khen hay chê, có việc cụ thể
nào làm được không. Bạn dựng ứng dụng trích xuất có cấu trúc bằng `qwen2.5:3b`: schema, prompt, kiểm hợp lệ (schema
**và** quy tắc nghiệp vụ), gọi lại khi sai, chấm điểm trên nhãn thật của UIT-VSFC (cảm xúc, chủ đề), tự chấm các trường
không có nhãn, ước lượng chi phí nếu chạy bằng API, và viết `REPORT.md`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Test gọi `check_record` và `score_batch` trên dữ liệu nhỏ; giữ nguyên
  tên, tham số, `EXTRACTION_SCHEMA`, `LABELS`, `TOPICS`.
- Phần còn lại (prompt, vòng gọi, chạy, giao diện nếu muốn) là thiết kế của bạn, trong cell `explore`.
- Cần Ollama với `qwen2.5:3b`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/11-llm-apis-prompting/exercises/advanced/02-feedback-extraction -q`

In [ ]:
# explore: tìm gốc repo, tải UIT-VSFC nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

VSFC = ensure("uit-vsfc")
CACHE = VSFC / "cache"
CACHE.mkdir(exist_ok=True)

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json

import jsonschema
import numpy as np

## 1. Schema và quy tắc nghiệp vụ

`EXTRACTION_SCHEMA` là hợp đồng giữa model và phần còn lại của hệ thống. Schema bắt được kiểu và giá trị cho phép;
nó **không** bắt được các ràng buộc giữa các trường. `check_record(record) -> list[str]` trả mọi vấn đề (rỗng là
hợp lệ):

1. lỗi schema: mỗi lỗi của `jsonschema.Draft202012Validator(EXTRACTION_SCHEMA).iter_errors(record)` một chuỗi bắt đầu
   bằng `"schema: "`; có lỗi schema thì trả luôn, không kiểm tiếp;
2. `"rule: suggestion"` nếu `actionable` là `true` mà `suggestion` là `null` hoặc chuỗi rỗng sau `strip`, **hoặc**
   `actionable` là `false` mà `suggestion` khác `null`;
3. `"rule: aspects"` nếu `aspects` có hai phần tử trùng nhau sau `strip().lower()`, hoặc có phần tử rỗng sau `strip`.

`record` không phải dict (ví dụ `None` khi JSON hỏng) thì trả `["schema: không phải object"]`.

In [ ]:
LABELS = ["negative", "neutral", "positive"]
TOPICS = ["lecturer", "training_program", "facility", "others"]
EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": LABELS},
        "topic": {"type": "string", "enum": TOPICS},
        "aspects": {"type": "array", "items": {"type": "string", "maxLength": 40}, "maxItems": 3},
        "actionable": {"type": "boolean"},
        "suggestion": {"type": ["string", "null"]},
    },
    "required": ["sentiment", "topic", "aspects", "actionable", "suggestion"],
    "additionalProperties": False,
}


def check_record(record) -> list[str]:
    """Lỗi schema ("schema: ...") rồi lỗi quy tắc ("rule: suggestion", "rule: aspects"); rỗng là hợp lệ."""
    # TODO
    raise NotImplementedError

## 2. Chấm theo nhãn thật

`score_batch(records, golds) -> dict`: `records[i]` là dict trích được (hoặc `None` khi không có output dùng được),
`golds[i]` là `{"sentiment": ..., "topic": ...}` từ UIT-VSFC. Trả dict với đúng các khoá:

| khoá | ý nghĩa |
| --- | --- |
| `n` | số câu |
| `valid_rate` | tỉ lệ câu có `check_record(record) == []` |
| `sentiment_acc`, `topic_acc` | tỉ lệ đúng; câu không hợp lệ tính là **sai** |
| `sentiment_macro_f1` | macro-F1 trên `LABELS` |
| `topic_macro_f1` | macro-F1 trên `TOPICS` |
| `actionable_rate` | tỉ lệ câu **hợp lệ** có `actionable` là `true`, trên số câu hợp lệ (0.0 nếu không có câu hợp lệ) |

Câu không hợp lệ (kể cả hợp lệ schema nhưng sai quy tắc) đóng góp `None` vào cả hai macro-F1: là false negative của
nhãn đúng, không là false positive của nhãn nào (như bài Intermediate 02). Mọi giá trị là `float`, trừ `n` là `int`.
`ValueError` nếu hai danh sách lệch độ dài hoặc rỗng.

In [ ]:
def score_batch(records: list[dict | None], golds: list[dict]) -> dict:
    """Chấm một lô bản ghi trích xuất theo nhãn thật (bảng ở trên)."""
    # TODO
    raise NotImplementedError

## 3. Ứng dụng của bạn

Thiết kế trong các cell `explore` (khung gợi ý, sửa tuỳ ý):

- **Prompt** mô tả từng trường bằng ví dụ ngắn; gửi `EXTRACTION_SCHEMA` qua `format` của Ollama (hoặc
  `llmkit.chat(..., json_schema=...)`).
- **Vòng gọi lại**: bản ghi sai schema hoặc sai quy tắc thì gửi lại kèm danh sách lỗi (bài Basic 03), tối đa 1 tới 2
  lần; đếm số câu cần gọi lại.
- **Chạy** trên 150 câu ghim (như Intermediate 02) hoặc lớn hơn, cache theo câu; ghi token vào/ra và giây.
- **Chấm** bằng `score_batch`, so ít nhất hai cấu hình (ví dụ: không schema, có schema, có schema + gọi lại).
- **Tự chấm 30 câu** cho `aspects`, `actionable`, `suggestion` (cột người chấm trong một CSV), đo tỉ lệ đồng ý.
- **Chi phí**: chi phí ước lượng cho 10 000 phản hồi mỗi học kỳ qua một model API, dùng hàm của bài Basic 02 và bảng
  giá có ngày kiểm (hoặc ghi "giả định").
- Tuỳ chọn: một giao diện nhỏ (Gradio, Streamlit) hoặc CLI đọc CSV phản hồi và xuất CSV kết quả.

In [ ]:
# explore: khung ứng dụng (viết tiếp)
SYSTEM_PROMPT = ""     # TODO (explore): mô tả từng trường
# TODO (explore): chạy trên tập ghim có cache, gọi lại khi check_record báo lỗi, score_batch, tự chấm 30 câu